# Daily forecast dataset — `daily_14.json`

Answers to the six questions. **v2** — fixes applied to the first draft; each fix is marked
`FIX:` in the code with a comment explaining what was wrong.

**About the data**

* JSONL, one line per city: `{"city": {...}, "time": <fetch ts>, "data": [<daily forecasts>]}`.
* Temperatures are **Kelvin**; every °C value below is converted explicitly.
* A record holds **16 or 17** daily entries, and the window differs between the two groups
  (16 entries start 2017‑03‑13 *or* 03‑14; 17 entries start 03‑13).
* **Placeholder days (important):** 16,960 entries have `temp.day == temp.min == temp.max`.
  They sit almost exclusively at the *last* position and are not real forecasts
  (e.g. Ayorou's final day is `27.0 / 27.0 / 27.0` while the rest sit at 35–41 °C).
  Question 5 averages over the forecast horizon, so these must be excluded or the ranking
  is biased. Q4 still counts them, because it asks for row counts.
* **`humidity == 0` is a missing-value sentinel**, not 0 % RH — it affects 81.6 % of rows.
  It is nulled out in Q6 rather than reported as a measurement.
* **Forecast payloads are heavily duplicated** across city ids: 22,635 cities share only
  5,341 distinct `data` blocks (one block is reused by 163 cities). Identical temperatures
  for different cities in Q5 are therefore a property of the source data, not a bug here.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.width", 120)

DATA_PATH = Path("/home/lbjga/projects/omnifold/daily_14.json")

raw = pd.read_json(DATA_PATH, lines=True)

print("JSONL records:", len(raw))
print("columns:", raw.columns.tolist())

JSONL records: 22635
columns: ['city', 'time', 'data']


## Setup — flatten the nested `city` object

`FIX:` the first draft renamed columns **by position**
(`df.columns = ["city_info", "timestamp", "forecast"]`). That silently mislabels data if the
source file's key order ever changes, and it raises `ValueError: Length mismatch` as soon as
re-run after another cell has added a column. Renaming **by name** is order-independent and
idempotent; `.assign()` instead of `df["col"] = ...` makes the cell safe to re-run too.

In [2]:
df = raw.rename(
    columns={"city": "city_info", "time": "timestamp", "data": "forecast"}
)

df = df.assign(
    city_id=df["city_info"].apply(lambda x: x["id"]),
    city=df["city_info"].apply(lambda x: x["name"]),
    country=df["city_info"].apply(lambda x: x["country"]),
)

# Every record must expose the keys the later cells rely on.
assert df[["city_id", "city", "country", "forecast"]].notna().all().all()
assert set(df.columns) == {
    "city_info", "timestamp", "forecast", "city_id", "city", "country"
}

display(df[["city_id", "city", "country", "timestamp"]].head())

print("forecast entries per record:", df["forecast"].apply(len).value_counts().to_dict())

,city_id,city,country,timestamp
0,1283240,Kathmandu,NP,1489465075
1,3632308,Merida,VE,1489465075
2,1280737,Lhasa,CN,1489465075
3,745042,İstanbul,TR,1489465075
4,3496831,Mao,DO,1489465075


forecast entries per record: {16: 15886, 17: 6749}


## 1. How many city records are in the file?

In [3]:
num_records = len(df)
num_unique_cities = df["city_id"].nunique()

print("Number of city records:", num_records)
print("Number of distinct city ids:", num_unique_cities)

# Both readings of the question give the same number, so the answer is unambiguous.
assert num_records == num_unique_cities

Number of city records: 22635
Number of distinct city ids: 22635


## 2. How many unique countries are represented?

In [4]:
num_countries = df["country"].nunique()

print("Number of unique countries:", num_countries)

# Guard the answer: a null or malformed code would be counted as a country.
bad = df.loc[~df["country"].str.fullmatch(r"[A-Z]{2}"), "country"]
print("codes that are not 2 uppercase letters:", bad.nunique())

Number of unique countries: 243


codes that are not 2 uppercase letters: 0


## 3. Which 10 countries have the most cities?

In [5]:
top_countries = (
    df["country"]
    .value_counts()
    .head(10)
    .rename_axis("country")
    .reset_index(name="num_cities")
)

display(top_countries)

print("cities in the top 10:", int(top_countries["num_cities"].sum()), "of", len(df))

,country,num_cities
0,US,2924
1,IN,2393
2,BR,1194
3,DE,985
4,RU,953
5,CN,788
6,JP,752
7,GB,672
8,IT,670
9,FR,610


cities in the top 10: 11941 of 22635


## 4. Explode each city's daily forecast array — how many total rows?

`FIX:` the first draft claimed the result is *one row per (city, country, date)*. It is not:
`explode` keys on the **city record**, so the grain is *one row per (city_id, date)*.
468 `(city, country)` pairs contain more than one distinct city (4 × `Albany/US`,
4 × `Arlington/US`, 3 × `Allende/MX`, …), so names collide on that grain. Both counts are
reported below so the answer is unambiguous.

In [6]:
forecast_df = df.explode("forecast").reset_index(drop=True)

# One pass over the nested dicts instead of five separate .apply() walks.
attrs = pd.DataFrame(
    [
        {
            "date": pd.to_datetime(f["dt"], unit="s").strftime("%Y-%m-%d"),  # YYYY-MM-DD
            "temp_day": f["temp"]["day"],
            "temp_min": f["temp"]["min"],
            "temp_max": f["temp"]["max"],
            "humidity": f["humidity"],
            "weather_description": f["weather"][0]["description"],
            # Placeholder day: all three temps identical -> not a real forecast.
            "is_placeholder": f["temp"]["day"] == f["temp"]["min"] == f["temp"]["max"],
        }
        for f in forecast_df["forecast"]
    ],
    index=forecast_df.index,
)

forecast_df = pd.concat([forecast_df, attrs], axis=1)

total_rows = len(forecast_df)
distinct_grain = forecast_df.drop_duplicates(["city", "country", "date"]).shape[0]
distinct_id_date = forecast_df.drop_duplicates(["city_id", "date"]).shape[0]

print("Total exploded rows (one per city-day):", total_rows)
print("Rows per record:", forecast_df.groupby("city_id").size().value_counts().to_dict())
print("Distinct (city, country, date):", distinct_grain)
print("Distinct (city_id, date):", distinct_id_date)
print("Rows duplicated on the (city, country, date) grain:",
      total_rows - distinct_grain)

# sum of per-record forecast lengths must equal the exploded row count
assert total_rows == df["forecast"].apply(len).sum()

display(forecast_df[["city", "country", "date", "temp_day"]].head())

Total exploded rows (one per city-day): 368909
Rows per record: {16: 15886, 17: 6749}
Distinct (city, country, date): 358817
Distinct (city_id, date): 368909
Rows duplicated on the (city, country, date) grain: 10092


,city,country,date,temp_day
0,Kathmandu,NP,2017-03-14,286.84
1,Kathmandu,NP,2017-03-15,278.34
2,Kathmandu,NP,2017-03-16,278.23
3,Kathmandu,NP,2017-03-17,283.83
4,Kathmandu,NP,2017-03-18,283.37


## 5. Top 10 hottest cities by mean `temp.day`

`FIX:` three problems in the first draft.

1. **Placeholder days were averaged in.** 16,960 entries (4.6 %) have
   `day == min == max` and sit at the end of the horizon; they drag the mean down by
   ~0.5–0.8 K. Only 6,749 cities have that 17th day, so the bias is unequal and it changed
   the ranking (see the comparison below).
2. **`groupby(["city", "country"])` merged distinct cities** — 468 name/country pairs hold
   more than one `city_id`, e.g. four different `Albany`s in the US averaged into one "city".
   Grouping by `city_id` keeps them apart.
3. Sorting on `temp_day` (Kelvin) while displaying °C was confusing; sort on °C.

The horizon is *still* not equal for every city after the cleanup (15 or 16 valid days), so the
next cell re-checks the ranking on a single common window.

In [7]:
valid = forecast_df[~forecast_df["is_placeholder"]]

print(f"placeholder days dropped: {len(forecast_df) - len(valid):,} "
      f"({1 - len(valid) / len(forecast_df):.1%})")
print("valid days per city:", valid.groupby("city_id").size().value_counts().to_dict())

per_city = (
    valid.groupby("city_id")
    .agg(
        city=("city", "first"),
        country=("country", "first"),
        n_days=("temp_day", "size"),
        mean_K=("temp_day", "mean"),
    )
)
per_city["mean_C"] = per_city["mean_K"] - 273.15

top10_hottest = (
    per_city.nlargest(10, "mean_C")[["city", "country", "mean_C", "n_days"]]
    .reset_index(drop=True)
    .round(2)
)

display(top10_hottest)

placeholder days dropped: 16,960 (4.6%)
valid days per city: {16: 12424, 15: 10211}


,city,country,mean_C,n_days
0,Dourbali,TD,39.81,16
1,Ayorou,NE,39.41,16
2,Damaturu,NG,39.36,16
3,Maiduguri,NG,39.35,16
4,Magumeri,NG,39.35,16
5,Niamey,NE,39.25,16
6,Massakory,TD,39.25,16
7,Geidam,NG,39.22,16
8,Potiskum,NG,39.20,16
9,Daura,NG,39.20,16


In [8]:
def ranking(frame, by):
    "Top 10 mean temp.day in °C for a given grouping."
    return (frame.groupby(by)["temp_day"].mean() - 273.15).nlargest(10).round(2)


# (a) what the first draft reported: every day, grouped by city name
naive = ranking(forecast_df, ["city", "country"])

# (b) corrected: valid days only, one common 15-day window (2017-03-14 .. 2017-03-28)
common = valid[valid["date"].between("2017-03-14", "2017-03-28")]
windowed = ranking(common, ["city", "country"])

print("(a) first draft — all days, grouped by name")
display(naive.rename("mean_C").reset_index())

print("(b) robust check — common 15-day window only")
display(windowed.rename("mean_C").reset_index())

print("Winner is stable across all three treatments:", naive.index[0])

(a) first draft — all days, grouped by name


,city,country,mean_C
0,Dourbali,TD,39.08
1,Magumeri,NG,38.75
2,Maiduguri,NG,38.75
3,Damaturu,NG,38.74
4,Massakory,TD,38.69
5,Ayorou,NE,38.68
6,Potiskum,NG,38.62
7,Bogo,CM,38.59
8,Geidam,NG,38.58
9,Bongor,TD,38.57


(b) robust check — common 15-day window only


,city,country,mean_C
0,Dourbali,TD,40.05
1,Ayorou,NE,39.66
2,Damaturu,NG,39.58
3,Magumeri,NG,39.55
4,Maiduguri,NG,39.55
5,Niamey,NE,39.52
6,Massakory,TD,39.51
7,Geidam,NG,39.50
8,Potiskum,NG,39.45
9,Bongor,TD,39.40


Winner is stable across all three treatments: ('Dourbali', 'TD')


## 6. Tidy table for San Francisco, California

`FIX:` `humidity` was reported as `0` for 13 of the 16 days. That is the API's
missing-value sentinel (81.6 % of the file is `0`), and 0 % RH is impossible on a
"moderate rain" day, so it is converted to `NaN`.

`San Francisco` occurs **6 times in 5 countries** (US, PH, AR, SV, CR) — the `country == "US"`
filter is required, and it selects the single US record. The data has no state/province field,
so country is the only available proxy for "California".

In [9]:
sf_all = forecast_df[forecast_df["city"] == "San Francisco"]
print("records named 'San Francisco':")
display(sf_all.groupby("country").size().rename("rows").reset_index())

sf = forecast_df[
    (forecast_df["city"] == "San Francisco") & (forecast_df["country"] == "US")
].copy()

assert sf["city_id"].nunique() == 1, "expected exactly one San Francisco/US record"

sf_tidy = pd.DataFrame(
    {
        "date": sf["date"],
        "temp_min": sf["temp_min"].sub(273.15).round(2),
        "temp_max": sf["temp_max"].sub(273.15).round(2),
        # 0 -> NaN: sentinel for "no humidity reading", not 0 % relative humidity
        "humidity": sf["humidity"].replace(0, float("nan")),
        "weather_description": sf["weather_description"],
    }
).sort_values("date").reset_index(drop=True)

display(sf_tidy)

print("rows:", len(sf_tidy),
      "| span:", sf_tidy["date"].min(), "->", sf_tidy["date"].max())
print("humidity missing:", int(sf_tidy["humidity"].isna().sum()), "of", len(sf_tidy))

records named 'San Francisco':


,country,rows
0,AR,16
1,CR,16
2,PH,32
3,SV,16
4,US,16


,date,temp_min,temp_max,humidity,weather_description
0,2017-03-13,9.81,18.95,67.0,sky is clear
1,2017-03-14,7.45,25.93,55.0,sky is clear
2,2017-03-15,8.63,22.75,56.0,sky is clear
3,2017-03-16,10.18,18.75,NaN,light rain
4,2017-03-17,6.50,18.56,NaN,light rain
5,2017-03-18,9.20,16.79,NaN,light rain
6,2017-03-19,10.50,18.08,NaN,light rain
7,2017-03-20,6.82,18.35,NaN,light rain
8,2017-03-21,4.57,16.67,NaN,sky is clear
9,2017-03-22,3.85,16.50,NaN,light rain


rows: 16 | span: 2017-03-13 -> 2017-03-28
humidity missing: 13 of 16


## Recap

| # | Question | Answer |
|---|---|---|
| 1 | city records | 22,635 (also 22,635 distinct city ids) |
| 2 | unique countries | 243 |
| 3 | top 10 countries | US, IN, BR, DE, RU, CN, JP, GB, IT, FR |
| 4 | exploded rows | 368,909 rows = one per (city_id, date); 358,817 on the (city, country, date) grain |
| 5 | top 10 hottest | see cell above — placeholder days excluded, grouped by `city_id` |
| 6 | San Francisco | 16 days, 2017‑03‑13 → 2017‑03‑28, humidity `NaN` where unread |

**Notes / caveats**

* `date` is the **UTC** date of a local-midnight instant (`dt` hours span 01–23), so labels for
  cities east of UTC can be one day earlier than the local calendar date. Irrelevant for
  San Francisco (UTC‑7) and for rankings, which do not use the date.
* Equal temperatures for different cities in Q5 are source-data duplication, not a join error.
* 4.6 % of forecast entries are placeholder days and 81.6 % of humidity readings are the
  `0` sentinel; both are handled explicitly above.

In [10]:
summary = pd.Series(
    {
        "Q1 city records": len(df),
        "Q1 distinct city ids": df["city_id"].nunique(),
        "Q2 unique countries": df["country"].nunique(),
        "Q3 countries in top 10": len(top_countries),
        "Q4 exploded rows": len(forecast_df),
        "Q4 distinct (city,country,date)": distinct_grain,
        "Q5 hottest city": f'{top10_hottest.iloc[0]["city"]} '
                           f'({top10_hottest.iloc[0]["country"]}), '
                           f'{top10_hottest.iloc[0]["mean_C"]} °C',
        "Q6 San Francisco rows": len(sf_tidy),
    },
    name="value",
)
display(summary.to_frame())

,value
Q1 city records,22635
Q1 distinct city ids,22635
Q2 unique countries,243
Q3 countries in top 10,10
Q4 exploded rows,368909
"Q4 distinct (city,country,date)",358817
Q5 hottest city,"Dourbali (TD), 39.81 °C"
Q6 San Francisco rows,16
